In [1]:
# @title Package
from natsort import natsorted
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchaudio
import math
from sklearn import svm

import torchvision
import torchvision.transforms as transforms
import torchaudio.models as audio_models

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

import time

lib_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Code'
os.chdir(lib_dir)
print('library directory: ' + lib_dir)
from modules.networks_clf import *
from modules.signal import spectro_norm, lfp_spectro
from modules.data import *
from modules.metrics import accu_fun

library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code


In [2]:
# @title Load device
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

!nvidia-smi -L


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-1ec7deec-daa9-42c8-75df-9cf853ac435d)


In [3]:
# Set the signal parameters
spectro_args = {
    'nfft':800,
    'power':1,
    'LFP_bound':[0, 500],
    'MUA_bound':[500, 2000],
    'spectro_img':[224, 28],
    'LFP_img':[56 * 4, 28],
    'MUA_img':[0, 28],
    'sampling_lfp':2500,
    'sampling_mua':5000,
    'Log':False,
}

dict_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project37/Data/dat'
acronym_list = acronym_list_gen(dict_dir)

In [4]:
def iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index0, coordinate_list):

    data_test = TensorDataset(brain_signal_lfp[test_ind,:], brain_region_index0[test_ind], coordinate_list[test_ind])

    test_iter = DataLoader(data_test, batch_size=128, shuffle=True)

    return test_iter

In [5]:
def iter_subject_gen(train_ind, valid_ind, test_ind, test_subject_ind, brain_signal_lfp, brain_region_index0, coordinate_list):

    test_ind_od = test_subject_ind

    data_test = TensorDataset(brain_signal_lfp[test_ind_od,:], brain_region_index0[test_ind_od], coordinate_list[test_ind_od])

    test_iter_od = DataLoader(data_test, batch_size=128, shuffle=True)

    return test_iter_od

In [6]:
# @title Load data
key = 'stimOff_times'
file_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/'
# torch.save(brain_signal_lfp, '/content/drive/MyDrive/Project/BrainRegionId/Project43/Result/brain_signal_lfp/brain_signal_lfp.pt')
brain_signal_lfp = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Project43/Result/brain_signal_lfp/brain_signal_lfp.pt')
list_dict = torch.load(file_dir + '/list_dict.pt', weights_only=False)
# list_dict_acronym_selec = torch.load(file_dir + '/list_dict_acronym_selec.pt')
brain_region_index = list_dict['brain_region_index']
brain_region_index_Cosmos = list_dict['brain_region_index_Cosmos']
coordinate_list = list_dict['coordinate_list']
community_dict = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/community_dict.pt', weights_only=False)

In [7]:
resolution = 0.4

In [8]:
communities_accu_indiv = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/communities_accu_indiv.pt', weights_only=False)
marked_community = []
for community in np.unique(community_dict[resolution]['communities_label']):
    df_indiv0 = communities_accu_indiv[resolution][communities_accu_indiv[resolution]['acronym_test'] == community]
    for classifier_name in ['AnyNet', 'ViT', 'RNN']:
        if df_indiv0[df_indiv0['model_type'] == classifier_name]['acu_test'].mean() > 0.60:
            marked_community.append(community)

marked_community = np.unique(np.array(marked_community))

In [9]:
marked_community

array([0, 8])

In [ ]:
acronym_communities_accu_div = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/acronym_communities_accu_indiv.pt', weights_only=False)
selec_br = []
ig_br = []
for model in ['AnyNet', 'ViT', 'RNN']:
    dict_model = acronym_communities_accu_div[resolution][acronym_communities_accu_div[resolution]['model_type'] == model]
    for acronym in acronym_list:
        if dict_model[dict_model['acronym_test'] == acronym]['acu_test'].mean() < 0.5:
            ig_br.append(acronym)

ig_br = np.unique(np.array(ig_br))
selec_br = np.setdiff1d(np.array(acronym_list), ig_br)

In [ ]:
ig_br

array(['ACAd2/3', 'AIp2/3', 'AIp5', 'AOBgl', 'AT', 'AUDd5', 'AUDd6a',
       'AUDp6a', 'AUDpo6b', 'AUDv6a', 'BLAa', 'CEAc', 'CLI', 'CS',
       'DG-mo', 'DR', 'DTN', 'ECT2/3', 'ECT5', 'ENTl2', 'ENTl3', 'ENTm1',
       'FRP1', 'FRP2/3', 'IA', 'ICd', 'ICe', 'LDT', 'NI', 'NLL', 'ORBl1',
       'ORBl2/3', 'ORBm1', 'ORBvl1', 'ORBvl2/3', 'PC5', 'PCG', 'PERI2/3',
       'PF', 'PG', 'PL1', 'PL2/3', 'PPN', 'PRNr', 'PSV', 'RSPagl6b',
       'SOCl', 'SPVC', 'SSp-bfd1', 'SSp-bfd2/3', 'SSs2/3', 'SSs4', 'SSs5',
       'TEa2/3', 'TEa4', 'TEa5', 'TEa6a', 'VISC5', 'VISal6a', 'VISal6b',
       'VISl5', 'VISpor1', 'VISrl2/3', 'VTN', 'VeCB', 'XII'], dtype='<U10')

In [ ]:
communities_accu = {}

for resolution in [0.4]:
    communities = community_dict[resolution]['communities']
    communities_label = community_dict[resolution]['communities_label']
    communities_acronym = community_dict[resolution]['communities_acronym']

    id_community_mapping = []
    for acronym in acronym_list:
        if acronym in communities_acronym:
            if communities_label[np.argwhere(communities_acronym == acronym).flatten()] in marked_community:
                if acronym in selec_br:
                    label_top = np.argwhere(marked_community == communities_label[np.argwhere(communities_acronym == acronym).flatten()]).flatten()
                    id_community_mapping.append(label_top)
                else:
                    id_community_mapping.append(np.array([-1]))
            else:
                id_community_mapping.append(np.array([-1]))

        else:
            id_community_mapping.append(np.array([-1]))

    id_community_mapping = np.array(id_community_mapping)

    brain_region_index_intersect = id_community_mapping[brain_region_index].squeeze(-1)
    dropout_index = np.argwhere(brain_region_index_intersect == np.array([-1]))[:, 0]

    brain_region_index_intersect = torch.tensor(brain_region_index_intersect, dtype=torch.long)


    subject_od_ind = torch.load(f'/content/drive/MyDrive/Project/BrainRegionId/Project44/Model/Allen/subject_od_ind_Allen_{key}{0}.pt', weights_only=False)
    subject_od_list = torch.load(f'/content/drive/MyDrive/Project/BrainRegionId/Project44/Model/Allen/subject_od_list_Allen_{key}{0}.pt', weights_only=False)
    train_ind, valid_ind, test_ind, test_subject_ind = dat_ind_gen(list_dict, subject_od_ind, key)

    train_ind = np.setdiff1d(train_ind, dropout_index)
    valid_ind = np.setdiff1d(valid_ind, dropout_index)
    test_ind = np.setdiff1d(test_ind, dropout_index)
    test_subject_ind = np.setdiff1d(test_subject_ind, dropout_index)

    ind_selec = []
    for comm_ii, comm in enumerate(marked_community):
        comm_ind_top = brain_region_index_intersect
        ind_top_len = len(torch.argwhere(comm_ind_top == comm_ii).flatten())

        ind_selec.append(torch.argwhere(comm_ind_top == comm_ii).flatten()[np.random.choice(ind_top_len, 1000, replace=False)])

    ind_selec = torch.concat(ind_selec).numpy()

    # train_ind = np.intersect1d(train_ind, ind_selec)
    valid_ind = np.intersect1d(valid_ind, ind_selec)
    test_ind = np.intersect1d(test_ind, ind_selec)
    test_subject_ind = np.intersect1d(test_subject_ind, ind_selec)
    print(len(train_ind))
    print(len(valid_ind))
    print(len(test_ind))

    test_iter_AnyNet = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)
    test_iter_ViT = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)
    test_iter_RNN = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)

    test_iter_od_AnyNet = iter_subject_gen(train_ind, valid_ind, test_ind, test_subject_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)
    test_iter_od_ViT = iter_subject_gen(train_ind, valid_ind, test_ind, test_subject_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)
    test_iter_od_RNN = iter_subject_gen(train_ind, valid_ind, test_ind, test_subject_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)

    ind = 0
    model_dir = f'/content/drive/MyDrive/Project/BrainRegionId/Science/Model/Community/Top'
    key = 'stimOff_times'

    Classifier_AnyNet = torch.load(model_dir + f'/AnyNet_L_AllenCommunity_{key}_{ind}.pth', weights_only=False).to(device)
    Classifier_ViT = torch.load(model_dir + f'/ViT_L_AllenCommunity_{key}_{ind}.pth', weights_only=False).to(device)
    Classifier_RNN = torch.load(model_dir + f'/RNN_L_AllenCommunity_{key}_{ind}.pth', weights_only=False).to(device)


    train_args = {
        'overfitting_thres':0.60,
        'lr':5e-4,
        'norm':True,
        'temp':[True, True],
        'epochs':50,

    }

    model_type = []
    acu_test = []
    iter_list = [test_iter_AnyNet, test_iter_ViT, test_iter_RNN]
    Classifier_name = ['AnyNet', 'ViT', 'RNN']
    for Classifier_ii, Classifier in enumerate([Classifier_AnyNet, Classifier_ViT, Classifier_RNN]):
        Classifier.eval()
        for x_test1, y_test, coordinate_test in iter_list[Classifier_ii]:
            if Classifier_name[Classifier_ii] == 'RNN':
                x_test = lfp_spectro(x_test1, spectro_args, train_args)
                y_test = y_test.to(device)
                py_test = Classifier(x_test.to(device).squeeze(1).permute(0, 2, 1))
                del x_test, x_test1
                acu_test.append(accu_fun(py_test, y_test))
                model_type.append(Classifier_name[Classifier_ii])

            elif Classifier_name[Classifier_ii] in ['Chance', 'Linear']:
                x_test = lfp_spectro(x_test1, spectro_args, train_args)
                y_test = y_test.to(device)
                py_test = Classifier(x_test.to(device).squeeze(1).flatten(start_dim=1))
                del x_test, x_test1
                acu_test.append(accu_fun(py_test, y_test))
                model_type.append(Classifier_name[Classifier_ii])

            else:
                x_test = lfp_spectro(x_test1, spectro_args, train_args)
                y_test = y_test.to(device)
                py_test = Classifier(x_test.to(device))
                del x_test, x_test1
                acu_test.append(accu_fun(py_test, y_test))
                model_type.append(Classifier_name[Classifier_ii])
        print(Classifier_ii)


    acu = pd.DataFrame({
        'acu_test': np.array(acu_test),
        'model_type': model_type,
    })

    model_type_od = []
    acu_test_od = []
    iter_list = [test_iter_od_AnyNet, test_iter_od_ViT, test_iter_od_RNN]
    Classifier_name = ['AnyNet', 'ViT', 'RNN']
    for Classifier_ii, Classifier in enumerate([Classifier_AnyNet, Classifier_ViT, Classifier_RNN]):
        Classifier.eval()
        for x_test1, y_test, coordinate_test in iter_list[Classifier_ii]:
            if Classifier_name[Classifier_ii] == 'RNN':
                x_test = lfp_spectro(x_test1, spectro_args, train_args)
                y_test = y_test.to(device)
                py_test = Classifier(x_test.to(device).squeeze(1).permute(0, 2, 1))
                del x_test, x_test1
                acu_test_od.append(accu_fun(py_test, y_test))
                model_type_od.append(Classifier_name[Classifier_ii])

            elif Classifier_name[Classifier_ii] in ['Chance', 'Linear']:
                x_test = lfp_spectro(x_test1, spectro_args, train_args)
                y_test = y_test.to(device)
                py_test = Classifier(x_test.to(device).squeeze(1).flatten(start_dim=1))
                del x_test, x_test1
                acu_test_od.append(accu_fun(py_test, y_test))
                model_type_od.append(Classifier_name[Classifier_ii])

            else:
                x_test = lfp_spectro(x_test1, spectro_args, train_args)
                y_test = y_test.to(device)
                py_test = Classifier(x_test.to(device))
                del x_test, x_test1
                acu_test_od.append(accu_fun(py_test, y_test))
                model_type_od.append(Classifier_name[Classifier_ii])
        print(Classifier_ii)


    acu_od = pd.DataFrame({
        'acu_test': np.array(acu_test_od),
        'model_type': model_type_od,
    })

    cross_dat = {}
    cross_dat['cross-trial'] = acu
    cross_dat['cross-subject'] = acu_od
    communities_accu[resolution] = cross_dat



1082909
133
127
0
1
2
0
1
2


In [ ]:
acu

,acu_test,model_type
0,0.070866,AnyNet
1,0.078740,ViT
2,0.055118,RNN


In [ ]:
torch.save(communities_accu, '/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/communities_accu_Top_resolution04.pt')

In [ ]:
from google.colab import runtime
runtime.unassign()